# TD5 - First Machine Learning Pipeline with KNN

In this TD, we build a first supervised machine learning pipeline using **K-Nearest Neighbors (KNN)**.

The objective is not to discover many machine learning algorithms. The objective is to understand the complete workflow:

1. define a prediction problem;
2. split the dataset into training, validation and test sets;
3. train a first KNN classifier;
4. understand why scaling matters for distance-based models;
5. choose the hyperparameter `K` using the validation set;
6. evaluate the final model once on the test set.

We will use the **Breast Cancer Wisconsin dataset**, available directly in `scikit-learn`.

> Important: this dataset is used here only as a simple binary classification dataset. We will not discuss medical interpretation.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Part 1. Load the dataset

We start with a dataset already available in `scikit-learn`.  
Each row corresponds to one observation. Each feature is a numerical measurement computed from a cell image.

The target has two possible classes:

- `malignant`
- `benign`

This is therefore a **binary classification** problem.

In [3]:
# Load the dataset as pandas DataFrames / Series
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Target names:", data.target_names)

X.head()

Shape of X: (569, 30)
Shape of y: (569,)
Target names: ['malignant' 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:
# Display the first rows with the target added
# This is only for inspection. We will keep X and y separated for modeling.

df = X.copy()
df["target"] = y

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [5]:
print("Class distribution:")
print(y.value_counts())

print("\nClass distribution in proportions:")
print(y.value_counts(normalize=True))

print("\nSummary statistics for the first 8 features:")
display(X.iloc[:, :8].describe())

Class distribution:
target
1    357
0    212
Name: count, dtype: int64

Class distribution in proportions:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64

Summary statistics for the first 8 features:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200


### Questions

Answer briefly.

1. What does one row represent in this dataset?
2. What is the prediction target?
3. Is this a classification or regression task? Why?
4. Are the classes perfectly balanced?
5. Do all features have the same numerical scale?

## Part 2. Train / validation / test split

We split the dataset before any preprocessing.

We use three subsets:

- **training set**: fit preprocessing and train the model;
- **validation set**: choose the hyperparameter `K`;
- **test set**: final evaluation, used only once at the end.

We use the following proportions:

- 60% training;
- 20% validation;
- 20% test.

In [ ]:
random_seed = 42

# First split: keep 20% of the data for final testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_seed,
    stratify=y
)

# Second split: 25% of the remaining 80% becomes validation
# Final proportions: 60% train, 20% validation, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    random_state=random_seed,
    stratify=y_train_val
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

In [ ]:
# Check that the class proportions are similar in the three subsets
split_summary = pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "validation": y_val.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
})

split_summary.index = data.target_names[split_summary.index]
split_summary

### Questions

1. Why do we use `stratify=y` in the split?
2. Why do we need a validation set?
3. Why should the test set not be used to choose `K`?

## Part 3. First KNN classifier without scaling

KNN predicts the label of a new observation by looking at the labels of its nearest neighbors.

We first train a simple KNN classifier with `K = 5`, without scaling the features.  
This is a useful baseline, but it is not the best workflow for KNN.

In [ ]:
# TODO 1: create a KNN classifier with 5 neighbors
knn = KNeighborsClassifier(n_neighbors=...)

# Fit on the training set only
knn.fit(X_train, y_train)

# Predict on the validation set
y_val_pred = knn.predict(X_val)

print("Validation accuracy without scaling:", accuracy_score(y_val, y_val_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_val_pred))
print("\nClassification report:")
print(classification_report(y_val, y_val_pred, target_names=data.target_names))

## Part 4. KNN with scaling using a Pipeline

KNN is based on distances. If one feature has very large values, it can dominate the distance.

Therefore, for KNN, we usually scale numerical features before computing distances.

We use a `Pipeline`:

```text
StandardScaler -> KNeighborsClassifier
```

The scaler is fitted on the training set only. The same transformation is then applied to the validation set.

In [ ]:
# TODO 2: complete the pipeline with StandardScaler and KNeighborsClassifier
clf = Pipeline([
    ("scaler", ...),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

clf.fit(X_train, y_train)
y_val_pred_scaled = clf.predict(X_val)

print("Validation accuracy with scaling:", accuracy_score(y_val, y_val_pred_scaled))
print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_val_pred_scaled))
print("\nClassification report:")
print(classification_report(y_val, y_val_pred_scaled, target_names=data.target_names))

In [ ]:
comparison = pd.DataFrame({
    "model": ["KNN without scaling", "KNN with scaling"],
    "validation_accuracy": [
        accuracy_score(y_val, y_val_pred),
        accuracy_score(y_val, y_val_pred_scaled)
    ]
})

comparison

### Questions

1. Did scaling change the validation accuracy?
2. Why is scaling especially important for KNN?
3. Why is `Pipeline` a safer practice than scaling the full dataset manually before the split?

## Part 5. Choose K using the validation set

`K` is a **hyperparameter**. It is chosen before final evaluation.

We compare several values of `K` using the validation set.  
The test set is still not used.

In [ ]:
candidate_k = [1, 3, 5, 7, 9, 11, 15, 21, 31]
validation_scores = []

for k in candidate_k:
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    clf.fit(X_train, y_train)
    y_val_pred = clf.predict(...)

    # TODO 3: compute the validation accuracy
    val_acc = ...
    validation_scores.append(val_acc)

results = pd.DataFrame({
    "K": candidate_k,
    "validation_accuracy": validation_scores
})

results

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(candidate_k, validation_scores, marker="o")
plt.xlabel("K")
plt.ylabel("Validation accuracy")
plt.title("Choosing K with the validation set")
plt.grid(True)
plt.show()

In [ ]:
# TODO 4: find the index of the best validation score
index_of_best_k = ...
best_k = candidate_k[index_of_best_k]

print("Best K:", best_k)
print("Best validation accuracy:", validation_scores[index_of_best_k])

### Questions

1. Which value of `K` gives the best validation accuracy?
2. What can happen when `K` is too small?
3. What can happen when `K` is too large?

## Part 6. Final evaluation on the test set

After choosing `K`, we train the final model on:

```text
training set + validation set
```

Then we evaluate once on the test set.

This gives the final performance estimate.

In [ ]:
# Combine training and validation data for the final model
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# TODO 5: use the best K found on the validation set
final_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=...))
])

final_clf.fit(X_train_final, y_train_final)

y_test_pred = final_clf.predict(X_test)

print("Final test accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("\nClassification report:")
print(classification_report(y_test, y_test_pred, target_names=data.target_names))

## Part 7. Short conclusion

Complete the following sentences.

1. The prediction task is: ...
2. The model used in this TD is: ...
3. The validation set is used to: ...
4. The test set is used to: ...
5. Scaling is important for KNN because: ...
6. The final test accuracy is: ...

## Optional extension. Distance-weighted KNN

In the previous models, all neighbors had the same weight.

We can also use distance-weighted KNN: closer neighbors have more influence.

Run the following cell if you have time.

In [ ]:
distance_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k, weights="distance"))
])

distance_clf.fit(X_train_final, y_train_final)
y_test_pred_distance = distance_clf.predict(X_test)

print("Test accuracy with distance-weighted KNN:", accuracy_score(y_test, y_test_pred_distance))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred_distance))
print("\nClassification report:")
print(classification_report(y_test, y_test_pred_distance, target_names=data.target_names))